## Finance RAG Assistant (Text Document Version)

A lightweight Retrieval-Augmented Generation (RAG) application that allows users to ask natural language questions about their personal finance documents stored locally. The app automatically loads text-based files from the user's Documents/finance_docs folder, splits them into chunks, converts them into vector embeddings using OpenAI’s embedding model, and stores them in a Chroma vector database. When a user asks a question through the Gradio chat interface, the system retrieves the most relevant document context from the vector database and combines it with the user’s prompt to generate an accurate answer using an LLM. This enables quick insights and question answering across financial reports, expense analyses, and other finance-related documents.

This version avoids PDF parsing issues and loads readable text documents (.txt, .md, .csv, etc).

In [ ]:
## DEPENDANCIES AND PACKAGES INSTALLATION
!pip install openai chromadb gradio python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 3.0 MB/s  0:00:023.1 MB/s eta 0:00:01:01


In [40]:
## IMPORTS

import os
import uuid
import chromadb
import gradio as gr
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI


In [ ]:
## SET UP OPENAI API KEY

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

if api_key:
    client = OpenAI(api_key=api_key)
else:
    raise ValueError("OPENAI_API_KEY is not set in the environment variables.")

In [ ]:
## SET UP DOCUMENT FOLDER AND CHROMA COLLECTION

DOCUMENT_FOLDER = Path.home() / "Documents" / "finance_docs"
CHROMA_COLLECTION = "finance_knowledge_base" 

In [ ]:
## ENSURE DOCUMENTS FOLDER EXISTS
def ensure_documents_folder():
    DOCUMENT_FOLDER.mkdir(parents=True, exist_ok=True)
    print("Place finance documents in:")
    print(DOCUMENT_FOLDER)

In [ ]:
## INITIALIZE VECTOR DATABASE

def initialize_vector_db():
    chroma_client = chromadb.Client()
    collection = chroma_client.get_or_create_collection(
        name=CHROMA_COLLECTION
    )
    return collection

In [ ]:
## LOAD DOCUMENTS

def load_documents():
    documents = []
    
    for file in DOCUMENT_FOLDER.iterdir():
        try:
            with open(file, "r", encoding="utf-8", errors="ignore") as f:
                text = f.read()
                
            if text.strip():
                documents.append({
                    "text": text,
                    "source": file.name
                })
        except Exception:
            print("Skipping file:", file.name)
    
    print("Documents loaded:", len(documents))
    return documents

In [ ]:
## CHUNK DOCUMENTS

def chunk_documents(documents, chunk_size=500):
    chunks = []
    
    for doc in documents:
        text = doc["text"]
        
        for i in range(0, len(text), chunk_size):
            chunk = text[i:i + chunk_size]
            
            chunks.append({
                "text": chunk,
                "source": doc["source"]
            })
    
    print("Chunks created:", len(chunks))
    return chunks

In [ ]:
## CREATE EMBEDDINGS

def create_embeddings(texts):
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=texts
    )
    
    embeddings = [item.embedding for item in response.data]
    return embeddings

In [ ]:
## STORE VECTORS
def store_vectors(chunks, embeddings, collection):
    ids = []
    documents = []
    
    for chunk in chunks:
        ids.append(str(uuid.uuid4()))
        documents.append(chunk["text"])
    
    collection.add(
        documents=documents,
        embeddings=embeddings,
        ids=ids
    )

In [42]:
## INITILIZE VECTOR DATABASE
collection = initialize_vector_db()

In [ ]:
## BUILD VECTOR DATABASE

def build_vector_database():
    documents = load_documents()
    
    if not documents:
        print("No documents found")
        return
    
    chunks = chunk_documents(documents)
    texts = [chunk["text"] for chunk in chunks]
    
    embeddings = create_embeddings(texts)
    store_vectors(chunks, embeddings, collection)
    
    print("Vector database built successfully")

In [ ]:
## EMBED QUERY

def embed_query(query):
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=query
    )
    
    return response.data[0].embedding

In [ ]:
## RETRIEVE CONTEXT

def retrieve_context(query_vector, collection, k=3):
    results = collection.query(
        query_embeddings=[query_vector],
        n_results=k
    )
    
    context = "\n\n".join(results["documents"][0])
    return context

In [ ]:
## SYSTEM PROMPT

SYSTEM_PROMPT = """
You are a financial analysis assistant.

Use the provided finance documents to answer the question.

If the answer is not in the documents,
say you do not know.
"""

In [ ]:
## GENERATE ANSWER

def generate_answer(question, context):
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT + "\n\nContext:\n" + context
        },
        {
            "role": "user",
            "content": question
        }
    ]
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages
    )
    
    return response.choices[0].message.content

In [ ]:
## INITILIZE VECTOR DATABASE
collection = initialize_vector_db()

In [ ]:
## CREATE RAG PIPELINE
def rag_pipeline(question):
    query_vector = embed_query(question)
    context = retrieve_context(query_vector, collection)
    answer = generate_answer(question, context)
    return answer

In [ ]:
## CREATE CHAT FUNCTION
def chat(message, history):
    answer = rag_pipeline(message)
    return answer

In [ ]:
## RUN PIPELINE
ensure_documents_folder()
build_vector_database()

Place finance documents in:
/home/steve/Documents/finance_docs
Documents loaded: 5
Chunks created: 5
Vector database built successfully


In [41]:
## GRADIO INTERFACE
view = gr.ChatInterface(
    chat,
    type="messages"
).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
